# Minutes-adjusted association — does the signal survive holding playing time equal?
_Read-only diagnostic: compares each signal's raw rank-association with points to its association after partialling out `minutes`, per position, with bootstrap CIs; then checks whether it holds across minutes bands. **Association only — no causal/predictive claim.** DGW excluded._

**Sections:** (a) raw vs minutes-adjusted rho (with CIs) · (b) does it hold across minutes bands?

---

## Setup
> Whole season, `minutes > 0`, **DGW excluded**; per position, leading-indicator universe (exact composites dropped). Adjusted rho via `partial_spearman` controlling for `minutes`, CI via `bootstrap_partial_rho`; raw CI via `bootstrap_spearman_ci`. A signal whose adjusted rho collapses toward 0 was largely a playing-time proxy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.relevance import (
    compute_relevance, leading_indicator_signals, leading_alive_signals, POSITIONS,
)
from research.kernels.inferential.resampling import (
    partial_spearman, bootstrap_partial_rho, bootstrap_spearman_ci,
)
from research.kernels.diagnostic.conditioning import (
    compute_conditional_rho, classify_heterogeneity,
)

POSITIONS = list(POSITIONS)
COLOURS = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}

try:
    _r = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _r = load_mart()

mart = _r.mart
df = mart[mart["gw"].between(1, _r.data_cutoff_gw)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()
df = df[df["is_dgw"] == False].copy()

leading = sorted(leading_indicator_signals(drop_exact_composites=True))
alive_by_pos = {}
for p in POSITIONS:
    rel = compute_relevance(df[df["position"] == p], signals=leading, group_cols=())
    alive_by_pos[p] = leading_alive_signals(rel)

print(f"Study range: GW 1 - {_r.data_cutoff_gw} | minutes > 0 | DGW excluded | n = {len(df):,}")
for p in POSITIONS:
    print(f"  {p}: n={len(df[df.position == p]):>6,} | {len(alive_by_pos[p])} live signals")

## (a) Raw vs minutes-adjusted rho
> The grey dot is the raw signal→points association; the coloured dot holds minutes equal. `shrinkage` (raw minus adjusted) is how much of the raw association was really just 'nailed-on starters play and score more'.

In [ ]:
def _arrs(d, sig):
    pair = d[[sig, "minutes", "total_points"]].dropna()
    return (pair[sig].to_numpy(float), pair["minutes"].to_numpy(float),
            pair["total_points"].to_numpy(float))

rows = []
for p in POSITIONS:
    d = df[df["position"] == p]
    for sig in alive_by_pos[p]:
        s, m, y = _arrs(d, sig)
        raw = bootstrap_spearman_ci(s, y)
        if raw is None:
            continue
        adj_rho, adj_lo, adj_hi = bootstrap_partial_rho(np.column_stack([s, m]), y, 0, partial_spearman)
        rows.append({
            "position": p, "signal": sig, "n": raw["n"],
            "rho_raw": raw["rho"], "raw_lo": raw["ci_lower"], "raw_hi": raw["ci_upper"],
            "rho_adj": round(adj_rho, 4), "adj_lo": round(adj_lo, 4), "adj_hi": round(adj_hi, 4),
            "shrinkage": round(raw["rho"] - adj_rho, 4),
        })
madj = pd.DataFrame(rows)
display(madj.sort_values(["position", "rho_raw"], ascending=[True, False]).round(4))

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharex=True)
for ax, p in zip(axes, POSITIONS):
    sub = madj[madj.position == p].sort_values("rho_raw")
    yv = np.arange(len(sub))
    ax.hlines(yv, sub["rho_adj"], sub["rho_raw"], color="lightgrey", zorder=1)
    ax.scatter(sub["rho_raw"], yv, color="#bdbdbd", label="raw", zorder=2)
    ax.scatter(sub["rho_adj"], yv, color=COLOURS[p], label="minutes-adjusted", zorder=3)
    ax.set_yticks(yv)
    ax.set_yticklabels(sub["signal"], fontsize=7)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(p)
    ax.set_xlabel("Spearman rho vs total_points")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Raw -> minutes-adjusted: signals collapsing toward 0 were largely a minutes proxy", y=1.03)
plt.tight_layout()
plt.show()

## (b) Does it hold across minutes bands?
> A signal that tracks points overall but reverses sign in a band is a trap. `compute_conditional_rho` reports rho within each minutes band (`1-29 / 30-59 / 60+`); `classify_heterogeneity` flags `heterogeneous_sign` (reverses) or `heterogeneous_magnitude` (holds direction, swings in strength).

In [ ]:
def _band(mn):
    return "1-29" if mn < 30 else "30-59" if mn < 60 else "60+"

dband = df.assign(band=df["minutes"].astype(int).map(_band))
het_rows = []
for p in POSITIONS:
    d = dband[dband["position"] == p]
    for sig in alive_by_pos[p]:
        cr = compute_conditional_rho(d, sig, "total_points", "band")
        row = {"position": p, "signal": sig, "verdict": classify_heterogeneity(cr)}
        for r in cr.itertuples():
            row[f"rho[{r.stratum}]"] = r.rho
        het_rows.append(row)
het_tbl = pd.DataFrame(het_rows)
display(het_tbl.sort_values(["position", "signal"]).round(4))
print("verdict counts:")
print(het_tbl["verdict"].value_counts().to_string())